In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.precision', 15)  # Increase decimal precision
pd.set_option('display.width', 150)     # Wider display
pd.set_option('display.max_columns', None)  # Show all column

# Phương pháp Dây cung (Secant Method)

# Điều kiện:

* $(a,b)$ là khoảng cách ly nghiệm

* $f'(x)$ xác định dấu không đổi trên $[a,b]$

* $f''(x)$ xác định dấu không đổi trên $[a,b]$

Điều kiện hội tụ:

* Chọn đúng điểm mốc $d$ (điểm Fourier - $f(d) \cdot f''(d) > 0$)
    
* Chọn đúng xuất phát ban đầu $x_0$ ($f(d) \cdot f(x_0) < 0$)

# Áp dụng

## 1. Hậu nghiệm
Đánh giá sai số theo 1 trong 2 công thức:
+ $|x_n - x^*| \leqslant \dfrac{|f(x_n)|}{m_1}$ $\quad\quad$(1)

+ $|x_n - x^*| \leqslant \dfrac{M_1 - m_1}{m_1} \cdot |x_n - x_{n-1}| $ $\quad\quad$(2)

với $m_1 = \min\limits_{[a,b]} |f'(x)|, \quad M_1 = \max\limits_{[a,b]} |f'(x)|$

### 1.1 Thuật toán
**Input:** Hàm số $f(x)$ và đạo hàm $f'(x)$, khoảng cách ly nghiệm $(a,b)$, sai số $\varepsilon$.

**Output:** Nghiệm xấp xỉ $x^*$.

**Thuật toán:**

1. **Khởi tạo:**
   - Nếu $f(a)f(b) > 0 \Rightarrow$ Dừng (khoảng không hợp lệ).
   - Nếu $f(a)=0$ (hoặc $f(b)=0$) $\Rightarrow x^*=a$ (hoặc $b$), dừng.
   - Tính $\displaystyle m_1 = \min_{[a,b]} |f'(x)|$ và $\displaystyle M_1 = \max_{[a,b]} |f'(x)|$.
   - Đặt $x_0 = a, d = b$, vòng lặp $i=0$.
2. **Quá trình lặp:**
   - Tính $\displaystyle x_{i+1} = \frac{d \cdot f(x_i) - x_i \cdot f(d)}{f(x_i) - f(d)}$.
   - Tại lần lặp đầu tiên ($i=0$): Nếu $f(a)f(x_{i+1}) < 0 \Rightarrow d = a$, ngược lại $\Rightarrow d = b$.
   - Tính sai số $\Delta_i$:
     + Theo công thức 1: $\displaystyle \Delta_i = \frac{|f(x_{i+1})|}{m_1}$.
     + Theo công thức 2: $\displaystyle \Delta_i = \frac{M_1 - m_1}{m_1}|x_{i+1} - x_i|$.
   - Nếu $f(x_{i+1}) = 0$ hoặc $\Delta_i < \varepsilon \Rightarrow x^* = x_{i+1}$, kết thúc thuật toán.
   - Gán $x_i \leftarrow x_{i+1}$, tăng $i \leftarrow i+1$.

### 1.2 Code
#### 1.2.1 Có cộng thêm sai số làm tròn
##### Công thức 1

In [13]:
def secant_iteration_v1 (f, df, a, b, n, rbl):	
	M1 = max([np.abs(df(x)) for x in [a, b]]) #M1 is the maximum value of |f'(x)| in the interval [a,b]
	m1 = min([np.abs(df(x)) for x in [a, b]]) #m1 is the minimum value of |f'(x)| in the interval [a,b]
	print(f"m1 = {m1}, M1 = {M1}")

	# Error function
	if (f(a) * f(b) >= 0) or (M1 * m1 <= 0): 
		print("Can not find a root in the given interval")
		return 0

	# Implementing Secant Method
	x = a
	mrk = b

	results = []
	results.append({
        'i': 0,
        'd': mrk,
        'x_i': x,
        'f(x_i)': f(x),
        'delta_x=|f(x_i)|/m_1': None
    })

	for i in range(n):
		# Calculate the next value of x
		x_new = (mrk * f(x) - x * f(mrk)) / (f(x) - f(mrk))
		delta_x = abs(f(x_new))/ m1

		results.append({
            'i': i + 1,
			'd': mrk,
            'x_i': x_new,
            'f(x_i)': f(x_new),
            'delta_x=|f(x_i)|/m_1': delta_x
        })
		
		# Prepare for next iteration
		x = x_new
		if (f(x_new) == 0): 
			break
		elif (i == 0):
			if f(a) * f(x_new) < 0:
				mrk = a
			else:
				mrk = b

	# Print the final result
	df_results = pd.DataFrame(results)
	print(df_results.to_string(index=False))

	if rbl == None:
		print(f"The value of root is: {x}")
	else:
		total_delta = delta_x + 0.5 * 10**(-rbl) #must calculate roundoff error
		print(f"The value of root with {rbl} decimal point is: {round(x, rbl)}")
		print(f"Relative error is: {total_delta}")

In [ ]:
f = lambda x: x ** 5 - 12; 
df = lambda x: 5 * (x ** 4)

a = 1
b = 2

n = 20
rbl = 9
secant_iteration_v1 (f, df, a, b, n, rbl)

#### Công thức 2

In [14]:
def secant_iteration_v2 (f, df, a, b, n, rbl):	
	M1 = max([np.abs(df(x)) for x in [a, b]]) #M1 is the maximum value of |f'(x)| in the interval [a,b]
	m1 = min([np.abs(df(x)) for x in [a, b]]) #m1 is the minimum value of |f'(x)| in the interval [a,b]
	print(f"m1 = {m1}, M1 = {M1}")

	# Error function
	if (f(a) * f(b) >= 0) or (M1 * m1 <= 0): 
		print("Can not find a root in the given interval");
		return 0;

	# Implementing Secant Method
	x = a; mrk = b;

	results = []
	results.append({
        'i': 0,
        'd': mrk,
        'x_i': x,
        'f(x_i)': f(x),
        'delta_x=(M1-m1)*|x_i-x_(i-1)|/m_1': None
    })

	for i in range(n):
		# Calculate the next value of x
		x_new = (mrk * f(x) - x * f(mrk)) / (f(x) - f(mrk))
		delta_x = (M1 - m1) * abs(x - x_new) / m1

		results.append({
            'i': i + 1,
			'd': mrk,
            'x_i': x_new,
            'f(x_i)': f(x_new),
            'delta_x=(M1-m1)*|x_i-x_(i-1)|/m_1': delta_x
        })
		
		# Prepare for next iteration
		x = x_new
		if (f(x_new) == 0): 
			break
		elif (i == 0):
			if f(a) * f(x_new) < 0:
				mrk = a
			else:
				mrk = b

	# Print the final result
	df_results = pd.DataFrame(results)
	print(df_results.to_string(index=False))

	if rbl == None:
		print(f"The value of root is: {x}")
	else:
		total_delta = delta_x + 0.5 * 10**(-rbl) #must calculate roundoff error
		print(f"The value of root with {rbl} decimal point is: {round(x, rbl)}")
		print(f"Relative error is: {total_delta}")

In [ ]:
f = lambda x: x ** 5 - 12; 
df = lambda x: 5 * (x ** 4)

a = 1
b = 2

n = 20
rbl = 9
secant_iteration_v2 (f, df, a, b, n, rbl)

### 1.2.2 Không cộng thêm sai số làm tròn

Đánh giá sai số theo 1 trong 2 công thức:
+ $|x_n - x^*| \leqslant \dfrac{|f(x_n)|}{m_1} < \epsilon$ $\quad\quad(1)$

+ $|x_n - x^*| \leqslant \dfrac{M_1 - m_1}{m_1} \cdot |x_n - x_{n-1}| < \epsilon$ $\quad\quad(2)$

với $m_1 = \min\limits_{[a,b]} |f'(x)|, \quad M_1 = \max\limits_{[a,b]} |f'(x)|$

#### Công thức 1

Từ công thức (1) suy ra Điều kiện dừng: $|f(x_n)| < m_1 \varepsilon $

In [2]:
def secant_recursion_absolute_v1 (f, df, a, b, eps):	
	M1 = max([np.abs(df(x)) for x in [a, b]]) #M1 is the maximum value of |f'(x)| in the interval [a,b]
	m1 = min([np.abs(df(x)) for x in [a, b]]) #m1 is the minimum value of |f'(x)| in the interval [a,b]
	print(f"m1 = {m1}, M1 = {M1}")

	# Error function
	if (f(a) * f(b) >= 0) or (M1 * m1 <= 0): 
		print("Can not find a root in the given interval")
		return 0

	# Implementing Secant Method
	x = a
	mrk = b 
	new_eps = m1*eps
	print(f"delta_x = {new_eps}")

	results = []
	results.append({
        'i': 0,
        'd': mrk,
        'x_i': x,
        'f(x_i)': f(x),
        'delta_x=|f(x_i)|': None
    })

	i = 0
	while True:
		# Calculate the next value of x
		x_new = (mrk * f(x) - x * f(mrk)) / (f(x) - f(mrk))
		current_delta_x = abs(f(x_new))

		results.append({
            'i': i + 1,
			'd': mrk,
            'x_i': x_new,
            'f(x_i)': f(x_new),
            'delta_x=|f(x_i)|': current_delta_x
        })
		
		# Prepare for next iteration
		x = x_new
		if (f(x_new) == 0): 
			break
		elif (i == 0):
			if f(a) * f(x_new) < 0:
				mrk = a
			else:
				mrk = b
				
        # Stopping condition
		if current_delta_x < new_eps:
			break
		else:
			i += 1

    # Print the final result
	df_results = pd.DataFrame(results) 
	print(df_results.to_string(index=False))

	print(f"The value of root with absolute error {eps} is: {x}")

Bài tập VD trên lớp:

In [ ]:
f = lambda x: np.exp(x)-4*np.cos(x/2)+1
df = lambda x: np.exp(x)+2*np.sin(x/2)

a = -3
b = -2

eps = 0.5 * pow(10, -7)

secant_recursion_absolute_v1 (f, df, a, b, eps)

Câu 8.1. Viết thuật toán tính gần đúng hằng số $e$ với sai số $\varepsilon$ cho trước. Áp dụng tính $e$ với bảy chữ số đáng tin sau dấu phẩy.

Câu 8.2. Viết thuật toán tính gần đúng hằng số $\pi$ với sai số $\varepsilon$ cho trước. Áp dụng tính $\pi$ với bảy chữ số đáng tin sau dấu phẩy.

Câu 9. Viết thuật toán tính gần đúng $\sqrt[n]{a}, a \in \mathbb{R}, n \in \mathbb{N}$ với sai số $\varepsilon$ cho trước. Áp dụng tính $\sqrt[5]{17}$ với 6 chữ số đáng tin.

Câu 10. Tìm nghiệm âm lớn nhất của phương trình $e^x-\cos 2x=0$ với 8 chữ số đáng tin bằng phương pháp dây cung.

In [3]:
f = lambda x: x**5 - 16*x - 32
df = lambda x: 5*x**4 - 16

a = 2
b = 4
eps = 2 * pow(10, -6)
secant_recursion_absolute_v1 (f, df, a, b, eps)

m1 = 64, M1 = 1264
delta_x = 0.000128
 i  d               x_i              f(x_i)   delta_x=|f(x_i)|
 0  4 2.000000000000000 -32.000000000000000                NaN
 1  4 2.066666666666667 -27.365727078189288 27.365727078189288
 2  4 2.122045534519685 -22.922515791072179 22.922515791072179
 3  4 2.167314670726936 -18.857003670296471 18.857003670296471
 4  4 2.203813269614244 -15.276500656041684 15.276500656041684
 5  4 2.232902775974285 -12.219276650759120 12.219276650759120
 6  4 2.255868322826373  -9.672927767054780  9.672927767054780
 7  4 2.273860587751530  -7.593585162203496  7.593585162203496
 8  4 2.287870502779402  -5.921856851694514  5.921856851694514
 9  4 2.298726856252361  -4.594141504620289  4.594141504620289
10  4 2.307107661162605  -3.549619422544460  3.549619422544460
11  4 2.313558335824401  -2.733915598020701  2.733915598020701
12  4 2.318512049333250  -2.100514821276846  2.100514821276846
13  4 2.322309478004550  -1.610821372786532  1.610821372786532
14  4 2.325216565

#### Công thức 2

Từ công thức (2) suy ra Điều kiện dừng: $|x_n - x_{n-1}| < \epsilon \dfrac{m_1}{M_1 - m_1}$

In [16]:
def secant_recursion_absolute_v2 (f, df, a, b, eps):	
	M1 = max([np.abs(df(x)) for x in [a, b]]) #M1 is the maximum value of |f'(x)| in the interval [a,b]
	m1 = min([np.abs(df(x)) for x in [a, b]]) #m1 is the minimum value of |f'(x)| in the interval [a,b]
	print(f"m1 = {m1}, M1 = {M1}")

	# Error function
	if (f(a) * f(b) >= 0) or (M1 * m1 <= 0): 
		print("Can not find a root in the given interval")
		return 0

	# Implementing Secant Method
	x = a; mrk = b; new_eps = eps*m1/(M1 - m1)
	print(f"delta_x = {new_eps}")

	results = []
	results.append({
        'i': 0,
        'd': mrk,
        'x_i': x,
        'f(x_i)': f(x),
        'delta_x=|x_i-x_(i-1)|': None
    })

	i = 0
	while True:
		# Calculate the next value of x
		x_new = (mrk * f(x) - x * f(mrk)) / (f(x) - f(mrk))
		current_delta_x = abs(x - x_new)

		results.append({
            'i': i + 1,
			'd': mrk,
            'x_i': x_new,
            'f(x_i)': f(x_new),
            'delta_x=|x_i-x_(i-1)|': current_delta_x
        })
		
		# Prepare for next iteration
		x = x_new
		if (f(x_new) == 0): 
			break
		elif (i == 0):
			if f(a) * f(x_new) < 0:
				mrk = a
			else:
				mrk = b
				
        # Stopping condition
		if current_delta_x < new_eps:
			break
		else:
			i += 1

    # Print the final result
	df_results = pd.DataFrame(results) 
	print(df_results.to_string(index=False))

	print(f"The value of root with absoulte error {eps} is: {x}")

Bài tập VD trên lớp:

In [ ]:
f = lambda x: np.exp(x)-4*np.cos(x/2)+1
df = lambda x: np.exp(x)+2*np.sin(x/2)

a = -3
b = -2

eps = 0.5 * pow(10, -7)

secant_recursion_absolute_v2 (f, df, a, b, eps)

Câu 8.1. Viết thuật toán tính gần đúng hằng số $e$ với sai số $\varepsilon$ cho trước. Áp dụng tính $e$ với bảy chữ số đáng tin sau dấu phẩy.

Câu 8.2. Viết thuật toán tính gần đúng hằng số $\pi$ với sai số $\varepsilon$ cho trước. Áp dụng tính $\pi$ với bảy chữ số đáng tin sau dấu phẩy.

Câu 9. Viết thuật toán tính gần đúng $\sqrt[n]{a}, a \in \mathbb{R}, n \in \mathbb{N}$ với sai số $\varepsilon$ cho trước. Áp dụng tính $\sqrt[5]{17}$ với 6 chữ số đáng tin.

Câu 10. Tìm nghiệm âm lớn nhất của phương trình $e^x-\cos 2x=0$ với 8 chữ số đáng tin bằng phương pháp dây cung.

In [17]:
f = lambda x: np.exp(x)-np.cos(2*x)
df = lambda x: np.exp(x)+2*np.sin(2*x)

a = -0.6
b = -0.3
eps = 0.5 * pow(10, -8)
secant_recursion_absolute_v2 (f, df, a, b, eps)

m1 = 0.38846672610835287, M1 = 1.315266535840426
delta_x = 2.0957423708397935e-09
 i    d                x_i             f(x_i)  delta_x=|x_i-x_(i-1)|
 0 -0.3 -0.600000000000000  0.186453881617353                    NaN
 1 -0.3 -0.393571608980660 -0.031228759902019      0.206428391019340
 2 -0.6 -0.423185833684605 -0.007747782099022      0.029614224703945
 3 -0.6 -0.430239932019379 -0.001720696181665      0.007054098334774
 4 -0.6 -0.431792243198658 -0.000372537831818      0.001552311179279
 5 -0.6 -0.432127654824050 -0.000080208811920      0.000335411625393
 6 -0.6 -0.432199839170246 -0.000017248565964      0.000072184346195
 7 -0.6 -0.432215360672904 -0.000003708274432      0.000015521502658
 8 -0.6 -0.432218697579106 -0.000000797198826      0.000003336906202
 9 -0.6 -0.432219414938725 -0.000000171378468      0.000000717359619
10 -0.6 -0.432219569153553 -0.000000036842131      0.000000154214827
11 -0.6 -0.432219602305919 -0.000000007920143      0.000000033152367
12 -0.6 -0.4322196094

### 1.2.3 Tìm nghiệm thoả mãn điều kiện sai số tương đối

Đánh giá sai số theo công thức: $\dfrac{|x_n - x^*|}{|x_n|} \leqslant \dfrac{M_1 - m_1}{m_1} \cdot \dfrac{|x_n - x_{n-1}|}{|x_n|} < \eta$

với $m_1 = \min\limits_{[a,b]} |f'(x)|, \quad M_1 = \max\limits_{[a,b]} |f'(x)|$


Từ công thức sai số hậu nghiệm, ta có Điều kiện dừng:  $\dfrac{|x_n - x_{n-1}|}{|x_n|} < \eta \dfrac{m_1}{M_1 - m_1}$

In [ ]:
def secant_recursion_relative (f, df, a, b, eta):	
	M1 = max([np.abs(df(x)) for x in [a, b]]) #M1 is the maximum value of |f'(x)| in the interval [a,b]
	m1 = min([np.abs(df(x)) for x in [a, b]]) #m1 is the minimum value of |f'(x)| in the interval [a,b]
	print(f"m1 = {m1}, M1 = {M1}")

	# Error function
	if (f(a) * f(b) >= 0) or (M1 * m1 <= 0): 
		print("Can not find a root in the given interval");
		return 0;

	# Implementing Secant Method
	x = a; mrk = b; new_eta = eta * m1/(M1 - m1);
	print(f"sigma_x = {new_eta}, m_1 = {m1}, M_1 = {M1}")

	results = []; i = 0;
	results.append({
        'i': 0,
        'd': mrk,
        'x_i': x,
        'f(x_i)': f(x),
        'sigma_x=|x_i-x_(i-1)|/|x_i|': None
    })
	while True:
		# Calculate the next value of x
		x_new = (mrk * f(x) - x * f(mrk)) / (f(x) - f(mrk))
		current_sigma_x = abs(x - x_new)/abs(x_new)

		results.append({
            'i': i + 1,
			'd': mrk,
            'x_i': x_new,
            'f(x_i)': f(x_new),
            'sigma_x=|x_i-x_(i+1)|/|x_i|': current_sigma_x
        })
		
		# Prepare for next iteration
		x = x_new
		if (f(x_new) == 0): 
			break
		elif (i == 0):
			if f(a) * f(x_new) < 0:
				mrk = a
			else:
				mrk = b
				
        # Stopping condition
		if current_sigma_x < new_eta:
			break
		else:
			i += 1

    # Print the final result
	df_results = pd.DataFrame(results) 
	print(df_results.to_string(index=False))

	print(f"The value of root with relative error {eta} is: {x}")

Câu 11. Giải phương trình với nghiệm có sai lệch không vượt quá 0.05% bằng phương pháp dây cung.

a. $x^5-3x^3+2x^2-x+5=0$

In [ ]:
f = lambda x: x**5 - 3*x**3 + 2*x**2 - x + 5
df = lambda x: 5*x**4 - 9*x**2 + 4*x - 1

a = -3
b = -2

eta = 0.05 * pow(10, -2)

secant_recursion_relative (f, df, a, b, eta)

b. $2^x-5x+\sin x = 0$

c. $e^{-x}-x=0$